# Known-input feature components for v005

Definition-only notebook: no assignment CSV reads, training, SSH, downloads, or submissions. Copy code cells tagged `v005-feature-definition` into the primary notebook. `make_sequence_features(frame, feature_mode)` returns `(X[breaths,80,F], names, breath_ids)` in the original breath and row order, with float32 finite features. The default `causal20` recipe copies the exact legacy `features` source from `sequence_refinement_v004b.ipynb`; offline/dynamics/combined append 14/13/27 columns, respectively.

Offline features use known future controls and the known `u_out` switch. Inspiratory summaries use only `u_out == 0` inputs. Dynamics features remain causal, use actual timestep differences, and are predictive proxies: valve-opening percentage is not measured flow. Pressure and ID values never enter a feature. R/C levels and 80-row contiguous breath structure are validated. Every feature is computed independently inside its breath, with no learned statistics or raw-frame mutation.

The final code cell defines `run_feature_component_checks_v005(project_root)` for synthetic CPU tests, including exact legacy-source and numerical comparisons. It does not run automatically. `research/feature_checks_v005.json` records the completed checks.


In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

STEPS = 80
V005_FEATURE_MODES = ('causal20', 'offline', 'dynamics', 'combined')
V005_BASE_FEATURE_NAMES = (
    'R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt',
    'control_sum', 'control_mean', 'control_integral',
    'u_in_lag_1', 'u_in_change_1', 'u_in_lag_2', 'u_in_change_2',
    'u_in_lag_4', 'u_in_change_4', 'u_in_lag_8', 'u_in_change_8',
    'u_in_mean_4', 'u_in_mean_8',
)
V005_OFFLINE_FEATURE_NAMES = (
    'u_in_lead_1', 'u_in_lead_2', 'u_in_lead_4', 'u_in_lead_8',
    'u_in_centered_mean_4', 'u_in_centered_mean_8',
    'inspiration_duration', 'inspiration_time_fraction',
    'time_to_u_out_switch', 'inspiration_u_in_peak',
    'inspiration_u_in_peak_time', 'inspiration_u_in_mean',
    'inspiration_u_in_std', 'inspiration_u_in_integral',
)
V005_EWM_TIMESCALES_SECONDS = (.05, .1, .2, .5, 1., 2.)
V005_DYNAMICS_FEATURE_NAMES = (
    'rc_tau_seconds', 'time_over_rc_tau', 'u_in_times_R', 'u_in_over_R',
    'du_in_dt', 'control_trapezoid_integral',
    'control_trapezoid_over_compliance_L',
    'control_ewm_tau_0_05', 'control_ewm_tau_0_1',
    'control_ewm_tau_0_2', 'control_ewm_tau_0_5',
    'control_ewm_tau_1', 'control_ewm_tau_2',
)


In [ ]:
def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
    """Return features in original row order, after verifying 80-row blocks."""
    if len(frame) % STEPS:
        raise ValueError("Unexpected breath length; do not reshape this file")
    blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
    breath_ids = blocks[:, 0]
    if not np.all(blocks == breath_ids[:, None]):
        raise ValueError("Breaths are not contiguous 80-row blocks")
    if len(np.unique(breath_ids)) != len(breath_ids):
        raise ValueError("Repeated breath blocks")
    if not frame["id"].is_unique:
        raise ValueError("Duplicate row IDs")
    u = frame["u_in"].to_numpy().reshape(-1, STEPS)
    t = frame["time_step"].to_numpy().reshape(-1, STEPS)
    if not np.all(np.diff(t, axis=1) > 0):
        raise ValueError("Timestamps must increase inside each breath")
    for column in ("R", "C"):
        values = frame[column].to_numpy().reshape(-1, STEPS)
        if not np.all(values == values[:, :1]):
            raise ValueError(f"{column} changes within a breath")

    columns = {}
    for column in ("R", "C", "time_step", "u_in", "u_out"):
        columns[column] = frame[column].to_numpy(dtype=np.float32)
    step = np.arange(STEPS)
    dt = np.diff(t, axis=1, prepend=t[:, :1])
    cumulative = np.cumsum(u, axis=1, dtype=np.float32)
    columns["step"] = np.broadcast_to(step, u.shape).ravel()
    columns["dt"] = dt.ravel()
    columns["control_sum"] = cumulative.ravel()
    columns["control_mean"] = (cumulative / (step + 1)).ravel()
    # Valve opening is not measured flow: this is a history proxy, not volume.
    columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
    for lag in (1, 2, 4, 8):
        shifted = np.empty_like(u)
        shifted[:, :lag] = u[:, :1]
        shifted[:, lag:] = u[:, :-lag]
        columns[f"u_in_lag_{lag}"] = shifted.ravel()
        columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
    padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
    for window in (4, 8):
        begin = np.maximum(0, step + 1 - window)
        columns[f"u_in_mean_{window}"] = (
            (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
        ).ravel()
    names = list(columns)
    matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
    if not np.isfinite(matrix).all() or "pressure" in names:
        raise ValueError("Invalid features")
    return matrix, names, breath_ids


In [ ]:
# Keep the legacy flat API available when importing these cells into a notebook.
features_causal20 = features


def _validate_feature_frame_v005(frame):
    """Validate known inputs only. Pressure and unrelated columns are never read."""
    if not isinstance(frame, pd.DataFrame):
        raise TypeError('Expected a pandas DataFrame')
    required = ('id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out')
    if not frame.columns.is_unique:
        raise ValueError('Duplicate column names')
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing required inputs: {sorted(missing)}')
    if not len(frame) or len(frame) % STEPS:
        raise ValueError('Expected nonempty contiguous 80-row breaths')
    for column in required:
        if not pd.api.types.is_numeric_dtype(frame[column]):
            raise ValueError(f'{column} must be numeric')
        values = frame[column].to_numpy()
        if not np.isfinite(values).all():
            raise ValueError(f'{column} must be finite')
    for column in ('id', 'breath_id'):
        values = frame[column].to_numpy()
        if not np.all(values == np.floor(values)):
            raise ValueError(f'{column} must contain integer IDs')
    if not frame['id'].is_unique:
        raise ValueError('Duplicate row IDs')
    blocks = frame['breath_id'].to_numpy().reshape(-1, STEPS)
    breath_ids = blocks[:, 0]
    if not np.all(blocks == breath_ids[:, None]):
        raise ValueError('Breaths are not contiguous 80-row blocks')
    if len(np.unique(breath_ids)) != len(breath_ids):
        raise ValueError('Repeated breath blocks')
    t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
    if not np.all(np.diff(t, axis=1) > 0):
        raise ValueError('Timestamps must increase inside each breath')
    for column, levels in (('R', (5, 20, 50)), ('C', (10, 20, 50))):
        values = frame[column].to_numpy().reshape(-1, STEPS)
        if not np.all(values == values[:, :1]):
            raise ValueError(f'{column} changes within a breath')
        if not np.isin(values, levels).all():
            raise ValueError(f'Unexpected {column} level')
    out = frame['u_out'].to_numpy().reshape(-1, STEPS)
    if not np.isin(out, (0, 1)).all():
        raise ValueError('u_out must be binary')
    # The approved data has a single inspiration followed by expiration. A fully
    # observed inspiration without an expiration sample is supported as well.
    if np.any(out[:, 0] != 0) or np.any(np.diff(out.astype(np.int8), axis=1) < 0):
        raise ValueError('Expected initial inspiration followed by at most one valve opening')
    return breath_ids


def feature_names_v005(feature_mode='causal20'):
    if feature_mode not in V005_FEATURE_MODES:
        raise ValueError(f'Unknown feature_mode: {feature_mode!r}')
    names = list(V005_BASE_FEATURE_NAMES)
    if feature_mode in ('offline', 'combined'):
        names.extend(V005_OFFLINE_FEATURE_NAMES)
    if feature_mode in ('dynamics', 'combined'):
        names.extend(V005_DYNAMICS_FEATURE_NAMES)
    return names


def _offline_feature_block_v005(frame):
    """Known whole-breath input context; no target values or statistics."""
    u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
    t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
    out = frame['u_out'].to_numpy().reshape(-1, STEPS)
    inspiration = out == 0
    n_breaths = len(u)
    columns = []
    for lead in (1, 2, 4, 8):
        shifted = np.empty_like(u)
        shifted[:, :-lead] = u[:, lead:]
        shifted[:, -lead:] = u[:, -1:]
        columns.append(shifted)
    prefix = np.concatenate((np.zeros((n_breaths, 1)), np.cumsum(u, axis=1)), axis=1)
    steps = np.arange(STEPS)
    for window in (4, 8):
        # Even centered windows have window//2 prior and window//2-1 future
        # neighbours, matching pandas rolling(center=True, min_periods=1).
        begin = np.maximum(0, steps - window//2)
        end = np.minimum(STEPS, steps + (window+1)//2)
        columns.append((prefix[:, end] - prefix[:, begin]) / (end-begin))
    has_expiration = np.any(out == 1, axis=1)
    first_expiration = np.argmax(out == 1, axis=1)
    switch_index = np.where(has_expiration, first_expiration, STEPS-1)
    switch_time = t[np.arange(n_breaths), switch_index]
    duration = switch_time - t[:, 0]
    # With strictly increasing 80-step timestamps and initial inspiration,
    # duration is positive, including the no-expiration boundary case.
    fraction = np.clip((t-t[:, :1])/duration[:, None], 0., 1.)
    until_switch = np.maximum(switch_time[:, None]-t, 0.)
    masked = np.where(inspiration, u, -np.inf)
    peak_index = np.argmax(masked, axis=1)  # first peak resolves a known-input tie
    peak = u[np.arange(n_breaths), peak_index]
    peak_time = t[np.arange(n_breaths), peak_index] - t[:, 0]
    count = inspiration.sum(axis=1)
    mean = np.sum(np.where(inspiration, u, 0.), axis=1)/count
    variance = np.sum(np.where(inspiration, (u-mean[:, None])**2, 0.), axis=1)/count
    std = np.sqrt(np.maximum(variance, 0.))
    # Integrate between inspiratory observations only; do not infer unobserved
    # flow or include the interval crossing into expiration.
    both_inspiration = inspiration[:, 1:] & inspiration[:, :-1]
    integral = np.sum(np.where(both_inspiration,
        .5*(u[:, 1:]+u[:, :-1])*np.diff(t, axis=1), 0.), axis=1)
    def broadcast(values):
        return np.broadcast_to(values[:, None], u.shape)
    columns.extend((broadcast(duration), fraction, until_switch, broadcast(peak),
        broadcast(peak_time), broadcast(mean), broadcast(std), broadcast(integral)))
    return np.stack(columns, axis=-1).astype(np.float32)


def _dynamics_feature_block_v005(frame):
    """Causal timing and control proxies; valve opening is not measured flow."""
    u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
    t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
    r = frame['R'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
    c_litres = frame['C'].to_numpy(dtype=np.float64).reshape(-1, STEPS)/1000.
    dt = np.diff(t, axis=1, prepend=t[:, :1])
    tau = r*c_litres  # R in cmH2O/(L/s), C converted from mL/cmH2O
    derivative = np.zeros_like(u)
    derivative[:, 1:] = np.diff(u, axis=1)/dt[:, 1:]
    trapezoid = np.zeros_like(u)
    trapezoid[:, 1:] = np.cumsum(.5*(u[:, 1:]+u[:, :-1])*dt[:, 1:], axis=1)
    # These interactions can help the learner represent R/C sensitivity. They
    # are not asserted to have physical pressure/flow/volume units because u_in
    # is a valve-opening percentage, not observed flow or driving pressure.
    columns = [tau, (t-t[:, :1])/tau, u*r, u/r, derivative,
               trapezoid, trapezoid/c_litres]
    scales = np.asarray(V005_EWM_TIMESCALES_SECONDS, dtype=np.float64)
    histories = np.empty((*u.shape, len(scales)), dtype=np.float32)
    state = np.broadcast_to(u[:, :1], (len(u), len(scales))).copy()
    histories[:, 0, :] = state
    # One loop over 80 timesteps; all breaths and timescales run together.
    # Irregular-time EWMA includes the control observed at the current step.
    for step in range(1, STEPS):
        alpha = np.exp(-dt[:, step, None]/scales[None, :])
        state = alpha*state + (1.-alpha)*u[:, step, None]
        histories[:, step, :] = state
    fixed = np.stack(columns, axis=-1).astype(np.float32)
    return np.concatenate((fixed, histories), axis=-1)


def make_sequence_features(frame, feature_mode='causal20'):
    """Return (X[breaths,80,F] float32, names, breath_ids), in input order.

    causal20: exact legacy features; offline: legacy + known future/context;
    dynamics: legacy + causal dynamics proxies; combined: all three blocks.
    No scaling, fitting, target reads, IDs as features, or frame mutation occur.
    """
    names = feature_names_v005(feature_mode)
    breath_ids = _validate_feature_frame_v005(frame)
    base_flat, base_names, old_ids = features_causal20(frame)
    if base_names != list(V005_BASE_FEATURE_NAMES) or not np.array_equal(breath_ids, old_ids):
        raise ValueError('Legacy feature schema changed')
    base = base_flat.reshape(-1, STEPS, len(base_names))
    if feature_mode == 'causal20':
        return base, names, breath_ids.copy()
    result = np.empty((len(breath_ids), STEPS, len(names)), dtype=np.float32)
    result[..., :len(base_names)] = base
    position = len(base_names)
    if feature_mode in ('offline', 'combined'):
        block = _offline_feature_block_v005(frame)
        result[..., position:position+block.shape[-1]] = block
        position += block.shape[-1]
    if feature_mode in ('dynamics', 'combined'):
        block = _dynamics_feature_block_v005(frame)
        result[..., position:position+block.shape[-1]] = block
        position += block.shape[-1]
    if position != len(names) or not np.isfinite(result).all():
        raise ValueError('Invalid feature output')
    return result, names, breath_ids.copy()


The six exponential histories use irregular-time EWMAs initialized at the first observed control, then `h_t = exp(-dt/tau)*h_(t-1) + (1-exp(-dt/tau))*u_t`. Timestep zero has zero derivative and zero accumulated integrals. The baseline's original rectangular integral is unchanged; the trapezoid integral is an additional feature.

Lead padding repeats the final known control. Even centered windows use two/four prior neighbours and one/three future neighbours for windows four/eight. Inspiration duration ends at the first `u_out == 1` timestamp. If all 80 samples are inspiration, the last observed timestamp supplies the observed boundary. Normalized inspiration time is clipped to [0,1], time to the switch is clipped below at zero, and peak time is relative to the breath's first timestamp. The full-inspiration integral joins only pairs of inspiratory observations. Standard deviation uses population normalization (`ddof=0`).


In [ ]:
def run_feature_component_checks_v005(project_root='.'):
    """Synthetic CPU checks only; return a report without fitting or data reads."""
    import ast
    import platform
    import sys
    from datetime import datetime, timezone

    checks = {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    def rejects(name, frame, mode='combined'):
        try:
            make_sequence_features(frame, mode)
        except (ValueError, TypeError):
            checks[name] = True
        else:
            raise AssertionError(name)
    def toy_frame(dtype=np.float32):
        steps = np.arange(STEPS, dtype=np.float64)
        frames = []
        for b, (r, c) in enumerate(((5, 10), (20, 20), (50, 50))):
            dt = .031 + .0001*((steps+b) % 7)
            t = np.cumsum(dt)-dt[0]
            u = 3.+b+2.*np.sin(steps/4.)+steps*.07
            out = (steps >= (29+b)).astype(np.int8)
            frames.append(pd.DataFrame(dict(id=np.arange(b*STEPS+1, (b+1)*STEPS+1),
                breath_id=np.full(STEPS, 300+b, dtype=np.int32),
                R=np.full(STEPS, r, dtype=np.int8), C=np.full(STEPS, c, dtype=np.int8),
                time_step=t.astype(dtype), u_in=u.astype(dtype), u_out=out,
                pressure=(steps*.3+b).astype(dtype))))
        return pd.concat(frames, ignore_index=True)

    root = Path(project_root)
    source_nb = json.loads((root/'sequence_refinement_v004b.ipynb').read_text())
    old_source = ''.join(source_nb['cells'][3]['source'])
    old_ast = ast.parse(old_source)
    old_node = next(n for n in old_ast.body if isinstance(n, ast.FunctionDef) and n.name == 'features')
    old_function_source = '\n'.join(old_source.splitlines()[old_node.lineno-1:old_node.end_lineno])+'\n'
    this_nb = json.loads((root/'research/feature_components_v005.ipynb').read_text())
    new_function_source = ''.join(next(c['source'] for c in this_nb['cells']
        if c.get('id') == 'v005-feature-baseline'))
    check('baseline_function_source_exact', old_function_source == new_function_source)
    old_namespace = dict(np=np, pd=pd, STEPS=STEPS)
    exec(compile(old_function_source, '<legacy-features>', 'exec'), old_namespace)
    shapes = {}
    for dtype in (np.float32, np.float64):
        frame = toy_frame(dtype)
        before = frame.copy(deep=True)
        old_x, old_names, old_ids = old_namespace['features'](frame)
        for mode, expected in (('causal20', 20), ('offline', 34), ('dynamics', 33), ('combined', 47)):
            x, names, ids = make_sequence_features(frame, mode)
            shapes[mode] = list(x.shape)
            check(f'{dtype.__name__}_{mode}_shape_dtype_finite',
                x.shape == (3, STEPS, expected) and x.dtype == np.float32 and np.isfinite(x).all())
            check(f'{dtype.__name__}_{mode}_legacy20_exact',
                np.array_equal(x[..., :20].reshape(-1, 20), old_x)
                and names[:20] == old_names and np.array_equal(ids, old_ids))
            check(f'{dtype.__name__}_{mode}_unique_names', len(names) == len(set(names)) == expected)
        pd.testing.assert_frame_equal(frame, before, check_exact=True)
        checks[f'{dtype.__name__}_raw_frame_unmutated'] = True

    frame = toy_frame()
    all_modes = {m: make_sequence_features(frame, m) for m in V005_FEATURE_MODES}
    changed = frame.copy(deep=True)
    changed['pressure'] = np.nan
    for mode, (x, names, ids) in all_modes.items():
        xx, nn, ii = make_sequence_features(changed, mode)
        check(f'{mode}_pressure_label_invariance', np.array_equal(xx, x) and names == nn and np.array_equal(ids, ii))
    changed = frame.copy(deep=True)
    changed['id'] += 1000000
    changed['breath_id'] += 10000
    for mode, (x, names, ids) in all_modes.items():
        xx, nn, ii = make_sequence_features(changed, mode)
        check(f'{mode}_id_value_invariance', np.array_equal(xx, x) and names == nn and np.array_equal(ids+10000, ii))
        singles = [make_sequence_features(frame.iloc[b*STEPS:(b+1)*STEPS], mode)[0] for b in range(3)]
        check(f'{mode}_cross_breath_independence', np.array_equal(np.concatenate(singles), x))
    reordered = pd.concat([frame.iloc[160:240], frame.iloc[0:80], frame.iloc[80:160]], ignore_index=True)
    for mode, (x, _, ids) in all_modes.items():
        xx, _, ii = make_sequence_features(reordered, mode)
        check(f'{mode}_whole_breath_permutation', np.array_equal(xx, x[[2, 0, 1]]) and np.array_equal(ii, ids[[2, 0, 1]]))

    changed = frame.copy(deep=True)
    positions = np.arange(len(frame)) % STEPS
    cutoff = 10
    changed.loc[positions > cutoff, 'u_in'] += np.float32(20.)
    changed.loc[positions > cutoff, 'time_step'] += np.float32(.001)
    for mode in ('causal20', 'dynamics'):
        xx, _, _ = make_sequence_features(changed, mode)
        check(f'{mode}_prefix_causal', np.array_equal(xx[:, :cutoff+1], all_modes[mode][0][:, :cutoff+1]))
    xx, names, _ = make_sequence_features(changed, 'offline')
    check('offline_intentional_future_input_access', not np.array_equal(xx[:, :cutoff+1, 20:], all_modes['offline'][0][:, :cutoff+1, 20:]))
    check('offline_legacy_prefix_still_causal', np.array_equal(xx[:, :cutoff+1, :20], all_modes['offline'][0][:, :cutoff+1, :20]))
    changed = frame.copy(deep=True)
    changed.loc[changed.u_out == 1, 'u_in'] = np.float32(900.)
    xx, names, _ = make_sequence_features(changed, 'offline')
    summary_indices = [names.index(n) for n in V005_OFFLINE_FEATURE_NAMES[6:]]
    check('inspiration_summaries_exclude_expiratory_controls',
        np.array_equal(xx[..., summary_indices], all_modes['offline'][0][..., summary_indices]))
    for window in (4, 8):
        actual = all_modes['offline'][0][..., all_modes['offline'][1].index(f'u_in_centered_mean_{window}')]
        expected = np.stack([frame.iloc[b*STEPS:(b+1)*STEPS].u_in.rolling(window, center=True, min_periods=1).mean().to_numpy() for b in range(3)]).astype(np.float32)
        check(f'centered_mean_{window}_matches_pandas', np.array_equal(actual, expected))

    # Analytic checks for constant controls and irregular timestamps.
    constant = frame.copy(deep=True)
    constant['u_in'] = np.float32(7.)
    x, names, _ = make_sequence_features(constant, 'combined')
    t = constant.time_step.to_numpy().reshape(3, STEPS).astype(np.float64)
    elapsed = t-t[:, :1]
    check('constant_control_trapezoid', np.allclose(x[..., names.index('control_trapezoid_integral')], 7.*elapsed, rtol=1e-6, atol=1e-6))
    check('constant_control_derivative_zero', np.array_equal(x[..., names.index('du_in_dt')], np.zeros((3, STEPS), np.float32)))
    check('constant_control_ewm_constant', np.array_equal(x[..., -6:], np.full((3, STEPS, 6), 7., np.float32)))
    check('rc_time_constant_units', np.allclose(x[:, 0, names.index('rc_tau_seconds')], [.05, .4, 2.5]))
    check('normalized_inspiration_time_bounds', np.all((x[..., names.index('inspiration_time_fraction')] >= 0) & (x[..., names.index('inspiration_time_fraction')] <= 1)))
    # In the absence of an expiration sample, last observed time is the boundary.
    all_inspiration = constant.copy(deep=True)
    all_inspiration['u_out'] = np.int8(0)
    ax, an, _ = make_sequence_features(all_inspiration, 'combined')
    check('all_inspiration_boundary_finite', np.isfinite(ax).all() and np.all(ax[:, -1, an.index('inspiration_time_fraction')] == 1))
    one_inspiration = constant.copy(deep=True)
    one_inspiration['u_out'] = np.tile(np.r_[np.int8(0), np.ones(STEPS-1, dtype=np.int8)], 3)
    ox, on, _ = make_sequence_features(one_inspiration, 'offline')
    check('single_inspiration_summary_boundary', np.all(ox[..., on.index('inspiration_u_in_std')] == 0) and np.all(ox[..., on.index('inspiration_u_in_integral')] == 0))

    rejects('reject_partial_breath', frame.iloc[:-1])
    rejects('reject_empty_frame', frame.iloc[:0])
    rejects('reject_missing_input', frame.drop(columns='u_in'))
    rejects('reject_unknown_mode', frame, 'not-a-mode')
    for name, column, value, row in (
        ('reject_duplicate_row_id', 'id', frame.id.iloc[1], 0),
        ('reject_noncontiguous_breath', 'breath_id', 999, 0),
        ('reject_nonincreasing_time', 'time_step', frame.time_step.iloc[0], 1),
        ('reject_changing_R', 'R', 50, 0),
        ('reject_changing_C', 'C', 50, 0),
        ('reject_nonfinite_control', 'u_in', np.nan, 0),
        ('reject_nonfinite_time', 'time_step', np.inf, 0),
        ('reject_nonbinary_u_out', 'u_out', 2, 0),
        ('reject_no_initial_inspiration', 'u_out', 1, 0),
        ('reject_u_out_reversal', 'u_out', 1, 2),
    ):
        invalid = frame.copy(deep=True)
        invalid.loc[row, column] = value
        rejects(name, invalid)
    repeated = pd.concat([frame.iloc[:80], frame.iloc[:80]], ignore_index=True)
    repeated['id'] = np.arange(1, len(repeated)+1)
    rejects('reject_repeated_breath_block', repeated)
    invalid = frame.copy(deep=True)
    invalid['R'] = np.int8(7)
    rejects('reject_unknown_R_level', invalid)
    invalid = frame.copy(deep=True)
    invalid['C'] = np.int8(7)
    rejects('reject_unknown_C_level', invalid)
    check('dynamics_block_shared_exactly', np.array_equal(all_modes['dynamics'][0][..., 20:], all_modes['combined'][0][..., 34:]))
    check('offline_block_shared_exactly', np.array_equal(all_modes['offline'][0][..., 20:], all_modes['combined'][0][..., 20:34]))
    return dict(scope='Synthetic CPU feature checks only; no assignment CSV reads, models, SSH, or submissions',
        created_at_utc=datetime.now(timezone.utc).isoformat(), interpreter=sys.executable,
        platform=platform.platform(), versions=dict(numpy=np.__version__, pandas=pd.__version__),
        passed=all(checks.values()), check_count=len(checks), checks=checks,
        feature_shapes=shapes, feature_modes={m:feature_names_v005(m) for m in V005_FEATURE_MODES},
        legacy_function_sha256=hashlib.sha256(old_function_source.encode()).hexdigest())
